In [1]:
# bootstrap_ci_map50_fieldplots.py
#
# Purpose: Bootstrap 95% CI for mAP50 on 50 field plot tiles
# Same methodology as test set bootstrap but on field plot evaluation
# Uses 390-tile model predictions vs v5 remapped labels (with corners)

import os
import numpy as np
import pandas as pd

SUMMER_PRED_DIR = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_v2\eval_summer\labels"
FALL_PRED_DIR   = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_v2\eval_fall\labels"
GT_DIR          = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_v2\labels"
OUTPUT_DIR      = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\statistical_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_BOOTSTRAP = 1000
IOU_THRESH  = 0.5
CLASSES     = [0, 1]
SEED        = 42


def compute_iou(pred, gt):
    def corners(b):
        return b[0]-b[2]/2, b[1]-b[3]/2, b[0]+b[2]/2, b[1]+b[3]/2
    px1,py1,px2,py2 = corners(pred)
    gx1,gy1,gx2,gy2 = corners(gt)
    ix1,iy1 = max(px1,gx1), max(py1,gy1)
    ix2,iy2 = min(px2,gx2), min(py2,gy2)
    if ix2<=ix1 or iy2<=iy1:
        return 0.0
    inter = (ix2-ix1)*(iy2-iy1)
    union = (px2-px1)*(py2-py1)+(gx2-gx1)*(gy2-gy1)-inter
    return inter/union if union>0 else 0.0


def compute_ap50(preds, gts_dict, n_gt):
    if n_gt==0 or not preds:
        return 0.0
    matched = {idx: [False]*len(boxes) for idx,boxes in gts_dict.items()}
    tp,fp = [],[]
    for conf,cx,cy,w,h,tile_idx in preds:
        gt_boxes = gts_dict.get(tile_idx,[])
        best_iou,best_j = 0.0,-1
        for j,gt in enumerate(gt_boxes):
            iou = compute_iou([cx,cy,w,h],gt)
            if iou>best_iou:
                best_iou,best_j = iou,j
        if best_iou>=IOU_THRESH and best_j>=0 and not matched[tile_idx][best_j]:
            tp.append(1); fp.append(0)
            matched[tile_idx][best_j] = True
        else:
            tp.append(0); fp.append(1)
    tp_cs = np.cumsum(tp)
    fp_cs = np.cumsum(fp)
    rec  = np.concatenate([[0], tp_cs/n_gt, [1]])
    prec = np.concatenate([[1], tp_cs/(tp_cs+fp_cs+1e-8), [0]])
    return float(np.trapezoid(prec, rec))


def load_data(pred_dir, gt_dir):
    gt_files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".txt")])
    print(f"  GT files: {len(gt_files)}")

    preds_by_class = {c:{} for c in CLASSES}
    gts_by_class   = {c:{} for c in CLASSES}

    for tile_idx, fname in enumerate(gt_files):
        for c in CLASSES:
            gts_by_class[c][tile_idx] = []
            preds_by_class[c][tile_idx] = []

        with open(os.path.join(gt_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts)<5: continue
                c = int(parts[0])
                if c in CLASSES:
                    gts_by_class[c][tile_idx].append(
                        [float(parts[1]),float(parts[2]),
                         float(parts[3]),float(parts[4])])

        pred_path = os.path.join(pred_dir, fname)
        if not os.path.exists(pred_path):
            continue
        with open(pred_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts)<6: continue
                c = int(parts[0])
                if c in CLASSES:
                    preds_by_class[c][tile_idx].append(
                        (float(parts[5]),float(parts[1]),float(parts[2]),
                         float(parts[3]),float(parts[4])))

    return preds_by_class, gts_by_class, len(gt_files)


def bootstrap_map50(preds_by_class, gts_by_class, n_tiles, label):
    np.random.seed(SEED)
    map50_samples = []
    print(f"\n  Running {N_BOOTSTRAP} bootstrap iterations on {n_tiles} tiles...")

    for i in range(N_BOOTSTRAP):
        if i%200==0:
            print(f"    Iteration {i}/{N_BOOTSTRAP}...")
        sampled = np.random.choice(n_tiles, size=n_tiles, replace=True)
        ap_scores = []
        for c in CLASSES:
            boot_preds = []
            boot_gts   = {}
            n_gt       = 0
            for new_idx,orig_idx in enumerate(sampled):
                boot_gts[new_idx] = gts_by_class[c].get(orig_idx,[])
                n_gt += len(boot_gts[new_idx])
                for conf,cx,cy,w,h in preds_by_class[c].get(orig_idx,[]):
                    boot_preds.append((conf,cx,cy,w,h,new_idx))
            boot_preds.sort(key=lambda x: x[0], reverse=True)
            ap_scores.append(compute_ap50(boot_preds, boot_gts, n_gt))
        map50_samples.append(np.mean(ap_scores))

    samples  = np.array(map50_samples)
    ci_lo    = np.percentile(samples, 2.5)
    ci_hi    = np.percentile(samples, 97.5)
    mean_boot= np.mean(samples)

    print(f"\n  {label} field plot bootstrap:")
    print(f"    Bootstrap mean mAP50: {mean_boot:.4f}")
    print(f"    95% CI:               [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"    CI width:             {ci_hi-ci_lo:.4f}")
    return mean_boot, ci_lo, ci_hi, samples


print("="*60)
print("BOOTSTRAP CI — SUMMER FIELD PLOTS (reported mAP50=0.5243)")
print("="*60)
preds_s,gts_s,n_s = load_data(SUMMER_PRED_DIR, GT_DIR)
mean_s,lo_s,hi_s,_ = bootstrap_map50(preds_s, gts_s, n_s, "Summer")

print("\n"+"="*60)
print("BOOTSTRAP CI — FALL FIELD PLOTS (reported mAP50=0.2285)")
print("="*60)
preds_f,gts_f,n_f = load_data(FALL_PRED_DIR, GT_DIR)
mean_f,lo_f,hi_f,_ = bootstrap_map50(preds_f, gts_f, n_f, "Fall")

print("\n"+"="*60)
print("SUMMARY")
print("="*60)
print(f"\nSummer field plots: mAP50=0.5243  95% CI=[{lo_s:.4f}, {hi_s:.4f}]")
print(f"Fall field plots:   mAP50=0.2285  95% CI=[{lo_f:.4f}, {hi_f:.4f}]")

pd.DataFrame({
    "season":         ["summer","fall"],
    "reported_mAP50": [0.5243, 0.2285],
    "bootstrap_mean": [round(mean_s,4), round(mean_f,4)],
    "ci_lower_95":    [round(lo_s,4),   round(lo_f,4)],
    "ci_upper_95":    [round(hi_s,4),   round(hi_f,4)],
    "ci_width":       [round(hi_s-lo_s,4), round(hi_f-lo_f,4)],
    "n_tiles":        [50, 50],
    "n_bootstrap":    [N_BOOTSTRAP, N_BOOTSTRAP],
}).to_csv(os.path.join(OUTPUT_DIR,"bootstrap_ci_fieldplots.csv"), index=False)
print(f"\nSaved: {OUTPUT_DIR}/bootstrap_ci_fieldplots.csv")

BOOTSTRAP CI — SUMMER FIELD PLOTS (reported mAP50=0.5243)
  GT files: 50

  Running 1000 bootstrap iterations on 50 tiles...
    Iteration 0/1000...
    Iteration 200/1000...
    Iteration 400/1000...
    Iteration 600/1000...
    Iteration 800/1000...

  Summer field plot bootstrap:
    Bootstrap mean mAP50: 0.5216
    95% CI:               [0.4051, 0.6222]
    CI width:             0.2171

BOOTSTRAP CI — FALL FIELD PLOTS (reported mAP50=0.2285)
  GT files: 50

  Running 1000 bootstrap iterations on 50 tiles...
    Iteration 0/1000...
    Iteration 200/1000...
    Iteration 400/1000...
    Iteration 600/1000...
    Iteration 800/1000...

  Fall field plot bootstrap:
    Bootstrap mean mAP50: 0.2297
    95% CI:               [0.1529, 0.3061]
    CI width:             0.1532

SUMMARY

Summer field plots: mAP50=0.5243  95% CI=[0.4051, 0.6222]
Fall field plots:   mAP50=0.2285  95% CI=[0.1529, 0.3061]

Saved: C:\Master Physische Geographie\Canada\Masterarbeit\results\statistical_analysis/b